In [ ]:
import pandas as pd
pd.set_option("display.float_format", "{:,.2f}".format)

In [9]:
ext = pd.read_csv(
    "data/raw/extracto_tarjetas_2005-09.csv",
    encoding="cp1252",   # el diccionario de letras antiguo
    sep=";",             # las columnas se separan con punto y coma
    skiprows=1,          # salta la primera línea (el comentario)
    skipfooter=1,        # salta la última línea (TOTAL REGISTROS)
    engine="python",     # skipfooter solo funciona con el lector "python"
)
ext.head()

ParserError: Expected 21 fields in line 2527, saw 23

In [10]:
print(lineas[2526])

6089;REVISAR; ver nota;90000;0;0;2;2;2;2;26928,00;29.773,00;30091,00;30922,00;34112,00;34.280,00;3600,00;1100,00;1630,00;4000,00;1200,00;0,00;si



In [11]:
filas_reparadas = []

def reparar_fila(campos):
    if len(campos) == 23 and campos[1].strip().upper() == "REVISAR":
        filas_reparadas.append(campos)     # apuntamos qué hemos tocado
        return campos[:1] + campos[3:]     # ID + todo lo que va después de la nota
    filas_rotas.append(campos)             # otro problema: mejor no inventar
    return None

filas_rotas = []
ext = pd.read_csv(
    "data/raw/extracto_tarjetas_2005-09.csv",
    encoding="cp1252", sep=";", skiprows=1, skipfooter=1,
    engine="python", on_bad_lines=reparar_fila,
)

print("Filas leídas:", len(ext), "· el sistema dijo: 30490")
print("Reparadas:", len(filas_reparadas), "· Apartadas:", len(filas_rotas))

Filas leídas: 30490 · el sistema dijo: 30490
Reparadas: 3 · Apartadas: 0


In [12]:
ext.columns.tolist()

['ID Cliente',
 'Límite Crédito',
 'estado_pago_sep',
 'estado_pago_ago',
 'estado_pago_jul',
 'estado_pago_jun',
 'estado_pago_may',
 'estado_pago_abr',
 'Importe Factura SEP',
 'Importe Factura AGO',
 'Importe Factura JUL',
 'Importe Factura JUN',
 'Importe Factura MAY',
 'Importe Factura ABR',
 'importe_pagado_sep',
 'importe_pagado_ago',
 'importe_pagado_jul',
 'importe_pagado_jun',
 'importe_pagado_may',
 'importe_pagado_abr',
 'Impago Mes Siguiente']

In [13]:
import unicodedata

def nombre_limpio(nombre):
    # quita las tildes: "Límite" -> "Limite"
    sin_tildes = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    # minúsculas, sin espacios al principio ni al final, y "_" en lugar de espacios
    return sin_tildes.strip().lower().replace(" ", "_")

ext.columns = [nombre_limpio(c) for c in ext.columns]
ext.columns.tolist()

['id_cliente',
 'limite_credito',
 'estado_pago_sep',
 'estado_pago_ago',
 'estado_pago_jul',
 'estado_pago_jun',
 'estado_pago_may',
 'estado_pago_abr',
 'importe_factura_sep',
 'importe_factura_ago',
 'importe_factura_jul',
 'importe_factura_jun',
 'importe_factura_may',
 'importe_factura_abr',
 'importe_pagado_sep',
 'importe_pagado_ago',
 'importe_pagado_jul',
 'importe_pagado_jun',
 'importe_pagado_may',
 'importe_pagado_abr',
 'impago_mes_siguiente']

In [14]:
ext.info()

<class 'pandas.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id_cliente            30490 non-null  int64
 1   limite_credito        30490 non-null  str  
 2   estado_pago_sep       30490 non-null  int64
 3   estado_pago_ago       30490 non-null  int64
 4   estado_pago_jul       30490 non-null  int64
 5   estado_pago_jun       30490 non-null  int64
 6   estado_pago_may       30490 non-null  int64
 7   estado_pago_abr       30490 non-null  int64
 8   importe_factura_sep   30181 non-null  str  
 9   importe_factura_ago   30163 non-null  str  
 10  importe_factura_jul   30178 non-null  str  
 11  importe_factura_jun   30155 non-null  str  
 12  importe_factura_may   30189 non-null  str  
 13  importe_factura_abr   30179 non-null  str  
 14  importe_pagado_sep    30200 non-null  str  
 15  importe_pagado_ago    30169 non-null  str  
 16  importe_pagado_

In [15]:
ext["limite_credito"].astype(float)

ValueError: could not convert string to float: '170.000,00'

In [16]:
def texto_a_numero(columna):
    texto = columna.astype(str).str.strip()             # 1. a texto y sin espacios
    texto = texto.str.replace("NT$", "", regex=False)   # 2. fuera la moneda
    texto = texto.str.strip()                           #    (y el espacio que deja)
    tiene_coma = texto.str.contains(",", regex=False)   # 3. ¿está en formato español?
    texto = texto.where(~tiene_coma, texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(texto, errors="coerce")         # 4. a número; lo que no se pueda, NaN

ext["limite_credito"] = texto_a_numero(ext["limite_credito"])
ext.loc[ext["limite_credito"] == -999, "limite_credito"] = float("nan")   # código de "sin dato"

ext["limite_credito"].describe()

count    3.041800e+04
mean     1.885201e+05
std      7.838318e+05
min      1.000000e+04
25%      5.000000e+04
50%      1.400000e+05
75%      2.400000e+05
max      5.000000e+07
Name: limite_credito, dtype: float64

In [19]:
ext.loc[ext["limite_credito"] > 1_000_000, "limite_credito"].sort_values().tolist()

[2000000.0,
 2000000.0,
 2000000.0,
 5000000.0,
 5000000.0,
 5000000.0,
 5000000.0,
 5000000.0,
 6000000.0,
 8000000.0,
 8000000.0,
 10000000.0,
 13000000.0,
 14000000.0,
 15000000.0,
 18000000.0,
 18000000.0,
 18000000.0,
 19000000.0,
 20000000.0,
 20000000.0,
 20000000.0,
 21000000.0,
 22000000.0,
 26000000.0,
 27000000.0,
 29000000.0,
 30000000.0,
 30000000.0,
 36000000.0,
 42000000.0,
 43000000.0,
 50000000.0,
 50000000.0]

In [20]:
imposibles = ext["limite_credito"] > 1_000_000
print("Límites corregidos (÷100):", imposibles.sum())
ext.loc[imposibles, "limite_credito"] = ext.loc[imposibles, "limite_credito"] / 100

ext["limite_credito"].describe()

Límites corregidos (÷100): 34


count      30,418.00
mean      167,560.18
std       129,846.17
min        10,000.00
25%        50,000.00
50%       140,000.00
75%       240,000.00
max     1,000,000.00
Name: limite_credito, dtype: float64

In [21]:
columnas_importes = [c for c in ext.columns if c.startswith("importe_")]
print(columnas_importes)

for col in columnas_importes:
    ext[col] = texto_a_numero(ext[col])

ext[columnas_importes].isna().sum()


['importe_factura_sep', 'importe_factura_ago', 'importe_factura_jul', 'importe_factura_jun', 'importe_factura_may', 'importe_factura_abr', 'importe_pagado_sep', 'importe_pagado_ago', 'importe_pagado_jul', 'importe_pagado_jun', 'importe_pagado_may', 'importe_pagado_abr']


importe_factura_sep    619
importe_factura_ago    650
importe_factura_jul    626
importe_factura_jun    678
importe_factura_may    584
importe_factura_abr    593
importe_pagado_sep     290
importe_pagado_ago     321
importe_pagado_jul     298
importe_pagado_jun     295
importe_pagado_may     289
importe_pagado_abr     288
dtype: int64

In [22]:
facturas = [c for c in columnas_importes if "factura" in c]
pagos    = [c for c in columnas_importes if "pagado" in c]

print("Facturas negativas:", (ext[facturas] < 0).sum().sum())
print("Pagos negativos:",    (ext[pagos] < 0).sum().sum())

Facturas negativas: 3919
Pagos negativos: 280


In [27]:
for col in pagos:
    negativos = ext[col] < 0
    ext.loc[negativos, col] = ext.loc[negativos, col].abs()

In [30]:
columnas_estado = [c for c in ext.columns if c.startswith("estado_pago_")]

for col in columnas_estado:
    ext[col] = ext[col].astype(str).str.strip().astype(int)

print(ext[columnas_estado].dtypes)
ext["estado_pago_sep"].value_counts().sort_index()

estado_pago_sep    int64
estado_pago_ago    int64
estado_pago_jul    int64
estado_pago_jun    int64
estado_pago_may    int64
estado_pago_abr    int64
dtype: object


estado_pago_sep
-2     2790
-1     5792
 0    14993
 1     3737
 2     2707
 3      328
 4       77
 5       26
 6       11
 7        9
 8       20
Name: count, dtype: int64

In [31]:
traduccion = {"1": 1, "si": 1, "sí": 1, "0": 0, "no": 0}
ext["impago_mes_siguiente"] = (
    ext["impago_mes_siguiente"].astype(str).str.strip().str.lower().map(traduccion)
)

sin_etiqueta = ext["impago_mes_siguiente"].isna()
print("Filas sin etiqueta (se quitan):", sin_etiqueta.sum())
ext = ext[~sin_etiqueta].copy()
ext["impago_mes_siguiente"] = ext["impago_mes_siguiente"].astype(int)

Filas sin etiqueta (se quitan): 95


In [32]:
print("Duplicados exactos:", ext.duplicated().sum())
ext = ext.drop_duplicates()
print("Filas después:", len(ext))
print("¿Algún cliente repetido aún?:", ext["id_cliente"].duplicated().sum())

Duplicados exactos: 446
Filas después: 29949
¿Algún cliente repetido aún?: 0


In [34]:
import json

with open("data/raw/clientes_crm.json", encoding="utf-8") as f:
    registros = json.load(f)

print("Registros:", len(registros))
print(registros[0])

crm = pd.json_normalize(registros)
crm.columns = [c.replace("datos_personales.", "") for c in crm.columns]
crm.head()

Registros: 30625
{'id_cliente': 'C-000526', 'datos_personales': {'sexo': 'F', 'fecha_nacimiento': '1978-01-09', 'estado_civil': 'soltero'}, 'nivel_estudios': 'Universidad', 'fecha_actualizacion': '2005-08-31'}


,id_cliente,nivel_estudios,fecha_actualizacion,sexo,fecha_nacimiento,estado_civil
0,C-000526,Universidad,2005-08-31,F,1978-01-09,soltero
1,C-011846,Universidad,2005-01-07,F,08/07/1978,Soltero/a
2,C-006403,Universidad,2005-07-31,mujer,1972-09-07,Soltero
3,C-014385,Bachillerato,2005-03-03,F,1980-02-27,casado
4,C-018448,Desconocido,2005-03-11,Mujer,1980-01-05,Casado/a


In [35]:
for col in ["sexo", "nivel_estudios", "estado_civil"]:
    print(col, crm[col].value_counts(dropna=False).to_dict())

print("IDs repetidos:", crm["id_cliente"].duplicated().sum())
print(crm["fecha_nacimiento"].sample(10, random_state=1).tolist())

sexo {'F': 14724, 'M': 9780, ' F ': 985, 'Mujer': 941, 'FEMENINO': 931, 'mujer': 916, 'H': 634, 'Hombre': 609, 'm': 581, 'MASCULINO': 524}
nivel_estudios {'Universidad': 11486, 'Postgrado': 8720, 'Bachillerato': 3963, 'Universitario': 984, 'universidad': 958, 'Univ.': 899, 'Máster/Doctorado': 733, 'postgrado': 681, 'POSTGRADO ': 674, 'bachillerato': 532, 'Secundaria': 521, 'Desconocido': 280, 'Otros': 100, '?': 47, 'otros': 24, nan: 14, '': 9}
estado_civil {'Soltero/a': 13053, 'Casado/a': 11122, 'Soltero': 1634, 'soltero': 1600, 'casado': 1448, 'CASADO': 1391, 'Otros': 261, nan: 45, 'otros': 37, 'Divorciado/a': 25, '': 9}
IDs repetidos: 600
['1960-10-15', '01/02/1971', '1977-09-24', '13/01/1980', '1961-03-24', '1975-04-01', '1965-12-12', '20/01/1981', '15/02/1968', '1960-07-23']


In [36]:
crm["id_cliente"] = crm["id_cliente"].str.replace("C-", "", regex=False).astype(int)

In [37]:
crm["fecha_actualizacion"] = pd.to_datetime(crm["fecha_actualizacion"])
crm = crm.sort_values("fecha_actualizacion").drop_duplicates("id_cliente", keep="last")
print("Clientes únicos:", len(crm))

Clientes únicos: 30025


In [38]:
sexo = {"M": "hombre", "H": "hombre", "HOMBRE": "hombre", "MASCULINO": "hombre",
        "F": "mujer", "MUJER": "mujer", "FEMENINO": "mujer"}

estudios = {"postgrado": "postgrado", "máster/doctorado": "postgrado",
            "universidad": "universidad", "univ.": "universidad", "universitario": "universidad",
            "bachillerato": "bachillerato", "secundaria": "bachillerato",
            "otros": "otros"}

estado_civil = {"casado/a": "casado", "casado": "casado",
                "soltero/a": "soltero", "soltero": "soltero",
                "otros": "otros", "divorciado/a": "otros"}

crm["sexo"] = crm["sexo"].str.strip().str.upper().map(sexo)
crm["nivel_estudios"] = crm["nivel_estudios"].str.strip().str.lower().map(estudios).fillna("desconocido")
crm["estado_civil"] = crm["estado_civil"].str.strip().str.lower().map(estado_civil).fillna("desconocido")

for col in ["sexo", "nivel_estudios", "estado_civil"]:
    print(col, crm[col].value_counts(dropna=False).to_dict())

sexo {'mujer': 18124, 'hombre': 11901}
nivel_estudios {'universidad': 14055, 'postgrado': 10585, 'bachillerato': 4917, 'desconocido': 345, 'otros': 123}
estado_civil {'soltero': 15989, 'casado': 13659, 'otros': 323, 'desconocido': 54}


In [39]:
formatos = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y"]
fecha = pd.Series(pd.NaT, index=crm.index)
for formato in formatos:
    intento = pd.to_datetime(crm["fecha_nacimiento"], format=formato, errors="coerce")
    fecha = fecha.fillna(intento)

FECHA_CORTE = pd.Timestamp("2005-09-30")
crm["edad"] = (FECHA_CORTE - fecha).dt.days // 365.25

imposibles = (crm["edad"] < 18) | (crm["edad"] > 100)
print("Edades imposibles (se dejan vacías):", imposibles.sum())
crm.loc[imposibles, "edad"] = float("nan")

crm = crm.drop(columns=["fecha_nacimiento", "fecha_actualizacion"])
crm["edad"].describe()

Edades imposibles (se dejan vacías): 148


count   29,666.00
mean        35.48
std          9.22
min         21.00
25%         28.00
50%         34.00
75%         41.00
max         79.00
Name: edad, dtype: float64

In [40]:
unida = ext.merge(
    crm,
    on="id_cliente",
    how="outer",
    indicator=True,
    validate="one_to_one",
)
unida["_merge"].value_counts()

_merge
both          29909
right_only      116
left_only        40
Name: count, dtype: int64

In [41]:
import os
os.makedirs("data/clean", exist_ok=True)

sin_pareja = unida[unida["_merge"] != "both"]
sin_pareja.to_csv("data/clean/revisar_sin_pareja.csv", index=False)
print("Apartados para revisar:", len(sin_pareja))

clientes = unida[unida["_merge"] == "both"].drop(columns="_merge")
print("Tabla final:", clientes.shape)

Apartados para revisar: 156
Tabla final: (29909, 25)


In [42]:
assert clientes["id_cliente"].is_unique, "Hay clientes repetidos"
assert clientes["impago_mes_siguiente"].isin([0, 1]).all(), "La etiqueta no es solo 0/1"
assert (clientes["limite_credito"].dropna() <= 1_000_000).all(), "Hay límites imposibles"
assert (clientes[pagos].fillna(0) >= 0).all().all(), "Hay pagos negativos"
assert clientes["sexo"].notna().all(), "Falta el sexo de algún cliente"
print("Todas las comprobaciones pasan")

print("Tasa de impago:", round(clientes["impago_mes_siguiente"].mean() * 100, 1), "%")

Todas las comprobaciones pasan ✅
Tasa de impago: 22.1 %


In [43]:
clientes.to_parquet("data/clean/clientes_modelo.parquet", index=False)
clientes.to_csv("data/clean/clientes_modelo.csv", index=False)

for fichero in ["clientes_modelo.parquet", "clientes_modelo.csv"]:
    tamano = os.path.getsize(f"data/clean/{fichero}") / 1_000_000
    print(f"{fichero}: {tamano:.1f} MB")

print(pd.read_parquet("data/clean/clientes_modelo.parquet").dtypes.value_counts())
print(pd.read_csv("data/clean/clientes_modelo.csv").dtypes.value_counts())

clientes_modelo.parquet: 1.8 MB
clientes_modelo.csv: 4.7 MB
float64    21
str         3
int64       1
Name: count, dtype: int64
float64    21
str         3
int64       1
Name: count, dtype: int64
